In [2]:
import tensorflow as tf

# Define inputs and trainable variables
x = tf.constant([[1.0, 2.0]])
W = tf.Variable([[0.5], [0.5]])
b = tf.Variable([0.1])

# Define the forward pass function
def forward(x):
    z = tf.matmul(x, W) + b
    return tf.nn.relu(z)

# Test the forward pass
output = forward(x)
print("Output:", output.numpy())

# Track operations for automatic differentiation
with tf.GradientTape() as tape:
    y = forward(x)
    loss = tf.reduce_mean(tf.square(y - 1.0))

# Calculate and print gradients with respect to W and b
grads = tape.gradient(loss, [W, b])
print("Gradient w.r.t. W:\n", grads[0].numpy())
print("Gradient w.r.t. b:", grads[1].numpy())


Output: [[1.6]]
Gradient w.r.t. W:
 [[1.2]
 [2.4]]
Gradient w.r.t. b: [1.2]


In [4]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load and preprocess the MNIST dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# Define the network architecture using the updated Keras standard
model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
model.fit(x_train, y_train, epochs=3, batch_size=32, validation_split=0.1)

# Evaluate on the test dataset
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"\nTest Accuracy: {test_acc:.4f}")


Epoch 1/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9084 - loss: 0.3164 - val_accuracy: 0.9633 - val_loss: 0.1351
Epoch 2/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9544 - loss: 0.1527 - val_accuracy: 0.9743 - val_loss: 0.0920
Epoch 3/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9648 - loss: 0.1152 - val_accuracy: 0.9705 - val_loss: 0.1005
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9680 - loss: 0.1044

Test Accuracy: 0.9680


In [9]:
import theano
import theano.tensor as T
import numpy as np
x = T.dmatrix('x')
W = T.dmatrix('W')
b = T.dvector('b')
z = T.dot(x, W) + b
y = T.nnet.relu(z)
loss = T.mean((y - 1) ** 2)
grad_W = T.grad(loss, W)
f = theano.function(inputs=[x, W, b], outputs=y)
grad_f = theano.function(inputs=[x, W, b], outputs=grad_W)
x_val = np.array([[1.0, 2.0]])
W_val = np.array([[0.5], [0.5]])
b_val = np.array([0.1])
print("Output:", f(x_val, W_val, b_val))
print("Gradient:", grad_f(x_val, W_val, b_val))

ModuleNotFoundError: No module named 'theano'

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

# Part 1: Manual Forward Pass and Gradients
x = torch.tensor([[1.0, 2.0]])
W = torch.tensor([[0.5], [0.5]], requires_grad=True)
b = torch.tensor([0.1], requires_grad=True)

z = torch.matmul(x, W) + b
y = torch.relu(z)
loss = torch.mean((y - 1.0) ** 2)

loss.backward()

print("Gradient w.r.t. W:\n", W.grad)
print("Gradient w.r.t. b:\n", b.grad)


# Part 2: Custom nn.Module Neural Network
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 4)
        self.fc2 = nn.Linear(4, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x))


# Initialize components
model = SimpleNet()
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# Single optimization step
output = model(x)
loss = criterion(output, torch.tensor([[1.0]]))

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("\nLoss after one step:", loss.item())


Gradient w.r.t. W:
 tensor([[1.2000],
        [2.4000]])
Gradient w.r.t. b:
 tensor([1.2000])

Loss after one step: 0.16078872978687286


In [10]:
Before looking at the code, here is what is happening mathematically in every single one of these scripts:
1. The Linear Equation (\(z = x \cdot W + b\)): The input data (\(x\)) is multiplied by the weights (\(W\)) and a bias (\(b\)) is added.
2. Activation Function (ReLU): The result \(z\) passes through a Rectified Linear Unit (ReLU). ReLU keeps positive numbers exactly as they are, and turns negative numbers into \(0\).
3. Loss Function (MSE): The code checks how far the network's output (\(y\)) is from our target goal (\(1.0\)) using Mean Squared Error.
4. Gradients (Derivatives): The code calculates "gradients". Gradients tell us the direction and amount we need to change our weights (\(W\)) and bias (\(b\)) to lower the error.

1. The TensorFlow Code (Dynamic Graph / Eager Execution)

TensorFlow 2.x uses Eager Execution, meaning operations happen immediately when you call them, just like regular Python.
• tf.constant and tf.Variable: Constants are fixed inputs (\(x\)). Variables (\(W\) and \(b\)) are parameters that the computer is allowed to change and train.
• tf.GradientTape() as tape: TensorFlow doesn't automatically calculate gradients because it takes up computer memory. The "Tape" acts like a digital recorder. It watches and records every math operation happening inside its block.
• tape.gradient(loss, [W, b]): The tape plays backward to calculate the exact derivatives (gradients) of the loss with respect to \(W\) and \(b\).

2. The Keras Code (High-Level Framework)

Keras sits on top of TensorFlow. Instead of making you write the math equations yourself, it lets you stack ready-made building blocks (layers).
• keras.datasets.mnist.load_data(): Downloads a famous dataset of 60,000 handwritten digits (0 through 9).
• x_train / 255.0: Normalises the images. Raw pixels range from 0 (black) to 255 (white). Dividing by 255 squishes them between 0.0 and 1.0, which helps neural networks learn much faster.
• keras.Sequential([...]): A linear stack of layers. Data flows from top to bottom.
	• Input(shape=(28, 28)): Tells the model to expect a 2D image grid.
	• Flatten(): Unrolls the 28x28 grid into a single long line of 784 pixels.
	• Dense(128, activation='relu'): A layer of 128 artificial neurons looking for patterns.
	• Dropout(0.2): Randomly turns off 20% of neurons during training. This forces the network to not rely too heavily on any single pixel (prevents overfitting).
• model.compile(...): Configures the learning strategy. It sets the brain/optimizer (adam) and how it measures mistakes (loss).
• model.fit(...): The actual training loop. It loops through the data 3 times (epochs) in groups of 32 images (batch_size).

3. The PyTorch Code (Dynamic / Imperative)

PyTorch is heavily used in research. It feels very similar to standard NumPy but with GPU support and automatic differentiation.
• requires_grad=True: This is PyTorch's version of the gradient tape. It tells PyTorch: "Keep an eye on this tensor, because I will need its gradient later."
• loss.backward(): This single command triggers the chain rule of calculus. It starts at the loss and calculates the gradients for every tensor that had requires_grad=True. The results are stored inside W.grad and b.grad.
• class SimpleNet(nn.Module): The standard way to build networks in PyTorch. You define the layers in __init__ and map out how data flows through them in the forward function.
• optimizer.zero_grad(): PyTorch accumulates (adds up) gradients by default. You must clear out old gradients before calculating new ones, otherwise your model will get confused.

4. The Theano Code (Static Symbolic Graph)

Theano behaves completely differently from the other three. It is a symbolic library. It doesn't actually calculate anything at first—it just builds a blueprint.
• T.dmatrix('x') and T.dvector('b'): These are not numbers. They are empty placeholders (like \(x\) and \(y\) in a school algebra equation).
• grad_W = T.grad(loss, W): Theano computes the derivative symbolically by looking at the mathematical graph blueprint, long before any data is passed.
• theano.function(...): This is the compilation step. Theano takes your abstract algebra blueprint and compiles it into a highly optimized, fast C-code function.
• f(x_val, W_val, b_val): Only at the very end do we pass actual numeric NumPy arrays into the compiled function to get a real answer

SyntaxError: invalid syntax (2159827153.py, line 1)